# Galea quality report

Interactive data-quality report for one Galea / OpenBCI GUI recording. Every number is measured from the data when the cells run.

* **Data:** set `DATA` below to a session folder, the folder holding `BrainFlow-RAW_*.csv` and the `OpenBCI-*.txt` files. If you leave it as `None`, the notebook uses `$GALEA_DATA`, or else the first session under `./data`.
* **Clock:** ExG (250 Hz) and Aux (50 Hz) share the headset clock. It is mapped to PC time by a linear fit to the lower envelope of the arrival timestamps. `t` = seconds since the first valid ExG sample.
* **Usable range:** `settle_time` is the later of two times: 15 s after the last railed sample, and the onset of stable PPG contact.
* Only raw signals are shown here. Derived signals (heart rate, breathing, head motion, EMG envelopes) are in `analysis.ipynb`.

In [ ]:
%load_ext autoreload
%autoreload 2
import pandas as pd
from IPython.display import Markdown, display
from galea import load_session, resolve_data, qc_report, channel_quality, settle_info
from galea import qc_plots as qp
from galea.browser import SignalBrowser

DATA = None          # session folder; None → $GALEA_DATA → first session under ./data
pd.set_option("display.max_colwidth", 400)

## 1. Session

In [ ]:
s = load_session(resolve_data(DATA))
si = settle_info(s)
qp.session_summary(s)

## 2. Findings
Automated checks, coloured by severity: red = error, yellow = warning, green = ok.

In [ ]:
q = qc_report(s)
q.style.apply(lambda r: [f"background-color: {qp.SEV_COLORS[r.severity]}"] * len(r), axis=1).hide(axis="index")

## 3. Clock
Offset between PC arrival time and device time.
* The fit (green) follows the per-window minima (black). Their residual is the fit error, and anything above the envelope is network or buffering latency.
* The vendor `Timestamp` column (red) is BrainFlow's running-mean sync, which has no drift model.
* Grey bands mark dropouts.

In [ ]:
display(pd.Series(s.clock.summary()).to_frame("value"))
qp.fig_clock(s)

## 4. Gaps & continuity

In [ ]:
s.gaps if s.gaps is not None and len(s.gaps) else Markdown("No dropouts.")

## 5. Railing & settling
"Railed" means |x| ≥ 90 % of the ADC full scale: ±375 000 µV for EEG (gain 12) and ±1 125 000 µV for EMG/EOG (gain 4).
* The dashed lines mark `settle_time` and the onset of PPG contact.
* The bar chart compares railing in the raw data with the GUI's signal-check estimate.

In [ ]:
qp.fig_railing_map(s).show()
qp.fig_railing_bars(s)

## 6. Channel quality after settling
Per ExG channel, measured over t > `settle_time`:
* **rms_1_40_uV:** 1–40 Hz RMS.
* **line_ratio:** 60 Hz PSD / 40–56 Hz PSD.
* **kurtosis:** excess kurtosis of the 1–40 Hz signal, which flags spikes and pops.
* **drift_uV:** 0.1–1 Hz SD.
* **grade:** a heuristic, relative within each group (see `galea.qc.channel_quality`).

In [ ]:
cq = channel_quality(s, si["settle"])
qp.style_grades(cq)

In [ ]:
qp.fig_psd(s, si["settle"])

## 7. Aux sensors
EDA, PPG and the raw IMU norms, all in file units (the IMU scales in the files are not physical).
* Movement during headset setup shows up in the accelerometer and gyro.
* PPG contact appears as a level jump.

In [ ]:
f = qp.fig_aux(s)
f if f is not None else Markdown("No Aux file.")

In [ ]:
c = qp.constant_channels(s)
Markdown("Constant Aux channels: none.") if c.empty else c

## 8. GUI signal-check
These are the OpenBCI GUI's live µVrms values, as logged. The GUI repeats rows, which are de-duplicated on load. Compare them with section 6.

In [ ]:
f = qp.fig_signalcheck(s)
f if f is not None else Markdown("No signal-check file.")

## 9. Browser (raw signals)
* Pick channels per group with Ctrl- or Shift-click; presets are in the dropdown.
* Zoom with box-zoom (double-click resets). Zooming re-fetches full-resolution data.
* Grey bands mark dropouts. With **blank railed samples** on, railed samples are hidden.
* **clip to window** restricts filters and scaling to the current view.

In [ ]:
b = SignalBrowser(s)
b

## 10. Summary

In [ ]:
Markdown(qp.recommendations(s, q, cq))